In [1]:
# python
import sys
import os
import importlib
# columnar analysis
from coffea.nanoevents import NanoEventsFactory
import awkward as ak
# local
sidm_path = str(sys.path[0]).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(llpnanoaodschema)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()

In [2]:
samples = [
    'DoubleMuon_2018C',
]
fileset = utilities.make_fileset(samples, "llpNanoAOD_v2", max_files=1, location_cfg="data_skimmed.yaml")
file_path = fileset[samples[0]]["files"][0]
print(file_path)

events = NanoEventsFactory.from_root(
    {file_path: "Events"},
    schemaclass=llpnanoaodschema.LLPNanoAODSchema,
).events()

root://xcache//store/group/lpcmetx/SIDM/Data/2018data/Skims/Run2018C_15Feb2022_UL2018/skimmed_output_1.root


/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(
/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(


In [3]:
first_events = events[:10]
muons = first_events.DSAMuon
print(muons[0])

[DSAMuon, DSAMuon]


In [4]:
nearest_muons, vals = muons.nearest(muons, return_metric=True)
print(muons.phi)
print(nearest_muons.phi)
print(vals)

[[-1.48, -1.21], [-2.77, -2.69, -2.79, ..., -1.93, -1.83], ..., [-1.45, 1.94]]
[[-1.48, -1.21], [-2.77, -2.69, -2.79, ..., -1.93, -1.83], ..., [-1.45, 1.94]]
[[0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0], ..., [...], [0, 0, 0, 0, 0, 0], [0, 0]]


In [5]:
import numpy as np
def cos_alpha(obj1, obj2):
    return np.cos(obj1.deltaangle(obj2))

In [6]:
cosmic_candidates, vals = muons.nearest(muons, metric=cos_alpha, return_metric=True)
for i in range(5):
    print(i)
    print(muons.phi[i])
    print(cosmic_candidates.phi[i])
    print(muons.deltaangle(cosmic_candidates)[i])
    print(np.cos(muons.deltaangle(cosmic_candidates))[i])
    print(vals[i])
    print()

0
[-1.48, -1.21]
[-1.21, -1.48]
[2.59, 2.59]
[-0.851, -0.851]
[-0.851, -0.851]

1
[-2.77, -2.69, -2.79, -2.83, -1.93, -1.83]
[-1.93, -1.93, -1.93, -1.93, -2.83, -2.83]
[0.401, 0.386, 0.42, 0.477, 0.477, 0.435]
[0.921, 0.926, 0.913, 0.888, 0.888, 0.907]
[0.921, 0.926, 0.913, 0.888, 0.888, 0.907]

2
[1.46, -1.44, 2.72]
[2.72, 2.72, -1.44]
[2.19, 2.3, 2.3]
[-0.58, -0.664, -0.664]
[-0.58, -0.664, -0.664]

3
[0.723, -2.75]
[-2.75, 0.723]
[2.81, 2.81]
[-0.946, -0.946]
[-0.946, -0.946]

4
[-1.75, 2.39, 0.157, -0.786]
[0.157, 0.157, 2.39, 2.39]
[2.57, 2.67, 2.67, 1.8]
[-0.841, -0.89, -0.89, -0.225]
[-0.841, -0.89, -0.89, -0.225]



In [7]:
cosmic_candidates, vals = muons.nearest(muons, metric=cos_alpha, threshold=-0.8, return_metric=True)
muons["cosmic_candidate"] = cosmic_candidates
muons["cos_alpha"] = vals

for i in range(5):
    print(i)
    print(muons.phi[i])
    print(cosmic_candidates.phi[i])
    print(vals[i])
    print("Possible cosmic?", ~ak.is_none(muons.cosmic_candidate, axis=1)[i])
    print()

0
[-1.48, -1.21]
[-1.21, -1.48]
[-0.851, -0.851]
Possible cosmic? [True, True]

1
[-2.77, -2.69, -2.79, -2.83, -1.93, -1.83]
[None, None, None, None, None, None]
[0.921, 0.926, 0.913, 0.888, 0.888, 0.907]
Possible cosmic? [False, False, False, False, False, False]

2
[1.46, -1.44, 2.72]
[None, None, None]
[-0.58, -0.664, -0.664]
Possible cosmic? [False, False, False]

3
[0.723, -2.75]
[-2.75, 0.723]
[-0.946, -0.946]
Possible cosmic? [True, True]

4
[-1.75, 2.39, 0.157, -0.786]
[0.157, 0.157, 2.39, None]
[-0.841, -0.89, -0.89, -0.225]
Possible cosmic? [True, True, True, False]

